importing necessary libraries and dataset

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob
import plotly.graph_objects as go

In [ ]:
df_imdb = pd.read_csv("/Users/abhimanyuchettiar/Downloads/imdb_top_1000.csv")

Data cleaning

In [ ]:

# 1. Clean 'Gross' column: Remove commas and convert to float
df_imdb['Gross'] = df_imdb['Gross'].str.replace(',', '').astype(float)

# 2. Clean 'Runtime': Remove ' min' and convert to integer
df_imdb['Runtime'] = df_imdb['Runtime'].str.replace(' min', '').astype(int)

# 3. Handle missing values
df_imdb['Meta_score'] = df_imdb['Meta_score'].fillna(df_imdb['Meta_score'].median())
df_imdb['Gross'] = df_imdb['Gross'].fillna(0) # Assume 0 if not disclosed
df_imdb['Certificate'] = df_imdb['Certificate'].fillna('Unrated')

# 4. Fix Released_Year (sometimes contains non-numeric strings in this dataset)
df_imdb['Released_Year'] = pd.to_numeric(df_imdb['Released_Year'], errors='coerce')
df_imdb.dropna(subset=['Released_Year'], inplace=True)
df_imdb['Released_Year'] = df_imdb['Released_Year'].astype(int)

Data analysis

IMDB Rating vs meta Score

In [ ]:
fig1 = px.scatter(df_imdb, x='IMDB_Rating', y='Meta_score', 
                 hover_name='Series_Title', color='IMDB_Rating',
                 title='IMDB Rating vs. Metascore',
                 labels={'IMDB_Rating': 'User Rating', 'Meta_score': 'Critic Score'},
                 template='plotly_dark')
fig1.show()

Gross Revenue over the years

In [ ]:
# Filter out 0 gross for better trend visualization
# 1. Group the data by year and sum the Gross revenue
gross_by_year = df_imdb[df_imdb['Gross'] > 0].groupby('Released_Year')['Gross'].sum().reset_index()

# 2. Sort by year to ensure the line flows chronologically
gross_by_year = gross_by_year.sort_values('Released_Year')

# 3. Plot the aggregated data
fig2 = px.area(gross_by_year, 
               x='Released_Year', 
               y='Gross', 
               title='Total Gross Revenue Trend of Top 1000 Movies (Aggregated)',
               color_discrete_sequence=['#E64A19'])

fig2.show()

Top 10 Directors by average IMDB Rating

In [ ]:
top_directors = df_imdb.groupby('Director')['IMDB_Rating'].mean().reset_index()
top_directors = top_directors.sort_values(by='IMDB_Rating', ascending=False).head(10)

fig3 = px.bar(top_directors, x='IMDB_Rating', y='Director', orientation='h',
             title='Top 10 Directors by Average Rating',
             color='IMDB_Rating', color_continuous_scale='Bluered')
fig3.update_layout(yaxis={'categoryorder':'total ascending'})
fig3.show()

Genre Popularity (Frequency vs Rating)

In [ ]:
# Explode genres as they are comma-separated strings
genres_split = df_imdb.assign(Genre=df_imdb['Genre'].str.split(', ')).explode('Genre')
genre_counts = genres_split['Genre'].value_counts().reset_index()
genre_counts.columns = ['Genre', 'Count']

fig4 = px.pie(genre_counts, values='Count', names='Genre', 
             title='Most Prevalent Genres in IMDb Top 1000',
             hole=0.4, color_discrete_sequence=px.colors.qualitative.Pastel)
fig4.show()

Distribution of Movie certificates

In [ ]:
# 1. Get counts and reset index
cert_counts = df_imdb['Certificate'].value_counts().reset_index()

# 2. Rename columns for clarity
cert_counts.columns = ['Certificate_Type', 'Movie_Count']

# 3. Create the Funnel using discrete color sequences
fig5 = px.funnel(cert_counts.head(8), 
                y='Certificate_Type', 
                x='Movie_Count',
                title='Distribution of Movie Certificates',
                # We use a sequential color list instead of a continuous scale
                color_discrete_sequence=px.colors.sequential.Tealgrn)
fig5.update_layout(showlegend=False)
fig5.show()

Runtime vs IMDB Rating (are longer movies better?)

In [ ]:
fig6 = px.density_heatmap(df_imdb, x='Runtime', y='IMDB_Rating', 
                         title='Heatmap: Runtime vs. IMDB Rating',
                         nbinsx=30, nbinsy=20, color_continuous_scale='Viridis')
fig6.show()